# Traffic-light CNN: train, check fixed-point accuracy, export ONNX (for Tensil)
Classes: **green / red / yellow** (alphabetical order is used everywhere). Input: 32x32 RGB (NHWC), pixel values scaled to 0..1.

Run top to bottom. Before CELL 1, upload `traffic_light_32.zip` (the zip from the cropping notebook) using the folder icon in the left sidebar. No Google Drive needed.

The final cell downloads `tensil_package.zip` with the ONNX model, test images and a results summary.

In [ ]:
# CELL 1: unzip the dataset
import os, zipfile, shutil
ZIP = '/content/traffic_light_32.zip'
assert os.path.exists(ZIP), 'Upload traffic_light_32.zip first (folder icon in the left sidebar).'
shutil.rmtree('/content/data', ignore_errors=True)
zipfile.ZipFile(ZIP).extractall('/content/data')
DATA = '/content/data'
if not os.path.isdir(f'{DATA}/train'):                    # zip had an extra top-level folder
    sub = [d for d in os.listdir(DATA) if os.path.isdir(f'{DATA}/{d}/train')]
    assert sub, 'Could not find a train/ folder inside the zip.'
    DATA = f'{DATA}/{sub[0]}'
print('data folder:', DATA, os.listdir(DATA))

In [ ]:
# CELL 2: install the export tools (torch is already in Colab)
!pip -q install onnx onnxruntime

In [ ]:
# CELL 3: load images
import glob, json, random, copy, csv
import numpy as np
from PIL import Image

CLASSES = sorted(os.listdir(f'{DATA}/train'))
print('classes (index order):', CLASSES)
assert CLASSES == ['green', 'red', 'yellow'], 'Expected exactly green/red/yellow folders'

def load_split(sp):
    X, y = [], []
    for i, c in enumerate(CLASSES):
        for f in sorted(glob.glob(f'{DATA}/{sp}/{c}/*.png')):
            im = Image.open(f).convert('RGB')
            if im.size != (32, 32):
                im = im.resize((32, 32))
            X.append(np.array(im, dtype=np.uint8)); y.append(i)
    return np.stack(X), np.array(y, dtype=np.int64)

Xtr_u8, ytr = load_split('train')
Xva_u8, yva = load_split('val')
Xte_u8, yte = load_split('test')
for n, (X, y) in {'train': (Xtr_u8, ytr), 'val': (Xva_u8, yva), 'test': (Xte_u8, yte)}.items():
    print(f'{n:<6}', X.shape, 'per class:', np.bincount(y, minlength=3).tolist())

In [ ]:
# CELL 4: model + training settings
import torch, torch.nn as nn, torch.nn.functional as F
torch.manual_seed(0); np.random.seed(0); random.seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

def to_tensor(X_u8):                                   # N,32,32,3 uint8 -> float 0..1, stays NHWC (like Tensil's examples)
    return torch.from_numpy(X_u8).float() / 255.0

Xtr, Xva, Xte = to_tensor(Xtr_u8), to_tensor(Xva_u8), to_tensor(Xte_u8)
Ytr, Yva, Yte = map(torch.from_numpy, (ytr, yva, yte))

class TLNet(nn.Module):
    # Mirrors the graph pattern of Tensil's own example models:
    #   Transpose(NHWC->NCHW) -> [Conv -> Relu -> MaxPool] x2 -> Conv -> Relu -> AveragePool(8x8) -> Reshape -> Gemm
    # The 8x8 average pool leaves a 1x1 map, so the Reshape has no layout ambiguity between PyTorch and Tensil.
    # Only layer types a small FPGA accelerator handles easily. No batch-norm, no softmax (output = raw logits).
    def __init__(self, n_classes=3):
        super().__init__()
        self.c1 = nn.Conv2d(3, 8, 3, padding=1)
        self.c2 = nn.Conv2d(8, 16, 3, padding=1)
        self.c3 = nn.Conv2d(16, 32, 3, padding=1)
        self.fc = nn.Linear(32, n_classes)
        self.pool = nn.MaxPool2d(2)
        self.gap = nn.AvgPool2d(8)
        self.relu = nn.ReLU()
    def forward(self, x):                              # x: N,32,32,3
        x = x.permute(0, 3, 1, 2)                      # -> N,3,32,32
        x = self.pool(self.relu(self.c1(x)))           # 16x16x8
        x = self.pool(self.relu(self.c2(x)))           # 8x8x16
        x = self.relu(self.c3(x))                      # 8x8x32
        x = self.gap(x)                                # 1x1x32
        return self.fc(x.reshape(-1, 32))

model = TLNet().to(device)
n_params = sum(p.numel() for p in model.parameters())
macs = 32*32*8*3*9 + 16*16*16*8*9 + 8*8*32*16*9 + 32*3
print(f'parameters: {n_params}   MACs per image: {macs}')

def augment(x):
    # simulates brightness / contrast / sensor noise / small position shifts (no hue change)
    B = x.size(0)
    m = x.mean((1, 2, 3), keepdim=True)
    c = torch.empty(B, 1, 1, 1, device=x.device).uniform_(0.8, 1.2)
    b = torch.empty(B, 1, 1, 1, device=x.device).uniform_(0.6, 1.4)
    x = ((x - m) * c + m) * b
    x = x + 0.02 * torch.randn_like(x)
    dx, dy = random.randint(-2, 2), random.randint(-2, 2)
    x = torch.roll(x, (dy, dx), (1, 2))
    return x.clamp(0, 1)

@torch.no_grad()
def predict(net, X, bs=256):
    net.eval(); out = []
    for i in range(0, len(X), bs):
        out.append(net(X[i:i+bs].to(device)).cpu())
    return torch.cat(out)

def accuracy(net, X, Y):
    return (predict(net, X).argmax(1) == Y).float().mean().item()

In [ ]:
# CELL 5: train (takes a few minutes on CPU, under a minute on GPU)
EPOCHS, BS, LR = 30, 64, 1e-3
opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, EPOCHS)
best_val, best_state, log = 0.0, None, []

for ep in range(1, EPOCHS + 1):
    model.train()
    perm = torch.randperm(len(Xtr))
    tot, n = 0.0, 0
    for i in range(0, len(Xtr), BS):
        idx = perm[i:i+BS]
        xb, yb = augment(Xtr[idx].to(device)), Ytr[idx].to(device)
        loss = F.cross_entropy(model(xb), yb)
        opt.zero_grad(); loss.backward(); opt.step()
        tot += loss.item() * len(idx); n += len(idx)
    sched.step()
    va = accuracy(model, Xva, Yva)
    log.append((ep, tot / n, va))
    if va >= best_val:
        best_val, best_state = va, copy.deepcopy(model.state_dict())
    print(f'epoch {ep:2d}  train loss {tot/n:.4f}  val acc {va:.4f}')

model.load_state_dict(best_state)
print('\nbest val acc:', round(best_val, 4))

In [ ]:
# CELL 6: test results (float32)
logits_te = predict(model, Xte)
pred = logits_te.argmax(1).numpy()
float_acc = float((pred == yte).mean())
cm = np.zeros((3, 3), dtype=int)
for t, p in zip(yte, pred):
    cm[t, p] += 1
print(f'TEST accuracy (float32): {float_acc:.4f}   ({len(yte)} images)')
print('\nconfusion matrix (rows = true, cols = predicted)')
print(' ' * 8 + ''.join(f'{c:>8}' for c in CLASSES))
for i, c in enumerate(CLASSES):
    print(f'{c:<8}' + ''.join(f'{v:>8}' for v in cm[i]))

In [ ]:
# CELL 7: estimate accuracy with 16-bit fixed point (weights and activations rounded to 1/256)
# This approximates how Tensil stores numbers. Tensil's own emulator is the real reference later.
FRAC = 8                                    # fraction bits (16-bit word, 8 integer bits incl. sign)
LO, HI = -2 ** (15 - FRAC), 2 ** (15 - FRAC) - 2 ** (-FRAC)

def fx(t):
    return torch.clamp(torch.round(t * 2 ** FRAC) / 2 ** FRAC, LO, HI)

class QWrap(nn.Module):
    def __init__(self, net):
        super().__init__()
        self.net = copy.deepcopy(net).cpu()
        with torch.no_grad():
            for p in self.net.parameters():
                p.copy_(fx(p))
        for m in self.net.modules():
            if isinstance(m, (nn.Conv2d, nn.Linear)):
                m.register_forward_hook(lambda mod, i, o: fx(o))
    def forward(self, x):
        return self.net(fx(x))

qmodel = QWrap(model).to(device)
fixed_acc = accuracy(qmodel, Xte, Yte)
print(f'float32 test accuracy : {float_acc:.4f}')
print(f'16-bit fixed (approx) : {fixed_acc:.4f}   (drop {100*(float_acc-fixed_acc):.2f} points)')

In [ ]:
# CELL 8: export to ONNX and check it matches PyTorch
# Expected ops: Transpose, Conv, Relu, MaxPool, AveragePool, Reshape, Gemm (all appear in Tensil's example models)
OPSET = 10        # Tensil's ResNet tutorial used an old opset; if `tensil compile` complains, try 11 or 12
import onnx, onnxruntime as ort

m_cpu = copy.deepcopy(model).cpu().eval()
dummy = torch.zeros(1, 32, 32, 3)                      # NHWC input, like Tensil's example models
kw = dict(input_names=['input'], output_names=['logits'], opset_version=OPSET)
try:
    torch.onnx.export(m_cpu, dummy, 'tl_cnn.onnx', dynamo=False, **kw)   # classic exporter
except TypeError:
    torch.onnx.export(m_cpu, dummy, 'tl_cnn.onnx', **kw)

om = onnx.load('tl_cnn.onnx'); onnx.checker.check_model(om)
print('ONNX ops used :', sorted({n.op_type for n in om.graph.node}))
print('input name    :', om.graph.input[0].name, '  output name:', om.graph.output[0].name)

sess = ort.InferenceSession('tl_cnn.onnx')
xs = Xte.numpy()
o = np.concatenate([sess.run(None, {'input': xs[i:i+1]})[0] for i in range(len(xs))])   # model batch size is fixed at 1
print('max |ONNX - PyTorch| logits difference:', float(np.abs(o - logits_te.numpy()).max()))
print('ONNX test accuracy:', float((o.argmax(1) == yte).mean()))

In [ ]:
# CELL 9: save everything for the Tensil step and download it
os.makedirs('pkg/demo_images', exist_ok=True)
shutil.copy('tl_cnn.onnx', 'pkg/tl_cnn.onnx')
torch.save(model.state_dict(), 'pkg/tl_cnn_weights.pt')
json.dump(CLASSES, open('pkg/class_names.json', 'w'))
np.save('pkg/test_images_uint8_NHWC.npy', Xte_u8)
np.save('pkg/test_labels.npy', yte)
np.save('pkg/test_logits_float.npy', logits_te.numpy())

rng = np.random.default_rng(0)
for ci, c in enumerate(CLASSES):                       # 6 demo images per class
    for k, i in enumerate(rng.choice(np.where(yte == ci)[0], 6, replace=False)):
        Image.fromarray(Xte_u8[i]).save(f'pkg/demo_images/{c}_{k}.png')

with open('pkg/training_log.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['epoch', 'train_loss', 'val_acc']); w.writerows(log)
json.dump({'classes': CLASSES, 'parameters': n_params, 'macs_per_image': macs,
           'test_images': int(len(yte)), 'float32_test_acc': float_acc,
           'fixed16_approx_test_acc': fixed_acc, 'confusion_matrix': cm.tolist(),
           'onnx_opset': OPSET, 'input': 'NHWC 1x32x32x3, pixels scaled to 0..1 (graph starts with a Transpose)',
           'output': 'raw logits, argmax = class'}, open('pkg/summary.json', 'w'), indent=2)

# inputs for the Tensil emulator: same layout as Tensil's demo file (1024 lines x 8 values; channels padded 3 -> 8)
os.makedirs('pkg/emu_inputs', exist_ok=True)
rng2 = np.random.default_rng(1)
emu_idx = [int(i) for c in range(3) for i in rng2.choice(np.where(yte == c)[0], 3, replace=False)]
with open('pkg/emu_inputs/reference.txt', 'w') as f:
    f.write(f'columns of the logits below: {CLASSES}\n')
    for k, i in enumerate(emu_idx):
        px = np.pad((Xte_u8[i].astype(np.float32) / 255.0).reshape(-1, 3), ((0, 0), (0, 5)))
        np.savetxt(f'pkg/emu_inputs/in_{k}.csv', px, delimiter=',', fmt='%.8f')
        lg = logits_te.numpy()[i]
        f.write(f'in_{k}.csv  true={CLASSES[yte[i]]}  float_logits={np.round(lg, 3).tolist()}  float_pred={CLASSES[int(lg.argmax())]}\n')

shutil.make_archive('/content/tensil_package', 'zip', 'pkg')
from google.colab import files
files.download('/content/tensil_package.zip')